# WALS-UD Language Feature Mapping Dataset

## Overview

This notebook demonstrates the creation of a curated mapping dataset linking Universal Dependencies (UD) treebanks to World Atlas of Language Structures (WALS) typological features.

The dataset maps **116 UD treebanks** to **5 WALS typological features**:
- **1A**: Order of Subject, Object and Verb
- **20A**: Fusion of Inflectional Morphology
- **26A**: Exponence of Selected Inflectional Categories
- **49A**: Number of Cases
- **51A**: Locus of Marking in the Clause

## What This Notebook Does

1. Loads the WALS-UD mapping data (mini version for demo)
2. Converts it to the experiment pipeline format
3. Displays summary statistics and visualizations

## Output Format

The output follows the `exp_sel_data_out.json` schema used in the experiment pipeline:
- Each UD treebank = one example
- `input`: JSON string with treebank metadata
- `output`: JSON string with WALS feature values
- `metadata_*`: Additional fields for analysis


In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Packages NOT pre-installed on Colab
# (loguru is not on the pre-installed list, so we install it)
_pip('loguru')

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'matplotlib==3.10.0')


In [ ]:
import json
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# loguru logger setup (same as original script)
try:
    from loguru import logger
    logger.remove()
    logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")
except ImportError:
    import logging
    logging.basicConfig(level=logging.INFO, format='%(asctime)s|%(levelname)s|%(message)s')
    logger = logging.getLogger(__name__)
    logger.info = logger.info
    logger.error = logger.error
    logger.warning = logger.warning


In [ ]:
# GitHub data URL for the mini demo data
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-6488ab-typological-predictors-of-dependency-dis/main/round-1/dataset-2/demo/mini_demo_data.json"

def load_data():
    """Load mini demo data from GitHub URL with local fallback."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception as e:
        print(f"GitHub load failed: {e}")
    # Local fallback
    if Path("mini_demo_data.json").exists():
        with open("mini_demo_data.json") as f:
            return json.load(f)
    # Try loading from wals_ud_mapping_mini.json (for demo purposes)
    if Path("wals_ud_mapping_mini.json").exists():
        with open("wals_ud_mapping_mini.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load data file")

print("Data loading helper defined.")


In [ ]:
# Load the WALS-UD mapping data
print("Loading WALS-UD mapping data...")
data = load_data()

# Handle both formats: direct mapping or experiment format
if 'mappings' in data:
    # Direct mapping format (wals_ud_mapping.json)
    mappings = data['mappings']
    metadata = data.get('metadata', {})
    print(f"Loaded {len(mappings)} UD treebank mappings (direct format)")
else:
    # Experiment format (mini_demo_data.json)
    # Extract from datasets[0].examples
    if 'datasets' in data:
        examples = data['datasets'][0]['examples']
        print(f"Loaded {len(examples)} examples (experiment format)")
        # Convert back to mapping format for processing
        mappings = {}
        for ex in examples:
            # Parse input and output JSON strings
            input_data = json.loads(ex['input'])
            output_data = json.loads(ex['output'])
            tb_name = input_data['ud_treebank']
            mappings[tb_name] = {
                'ud_treebank': tb_name,
                'ud_language_code': input_data['ud_language_code'],
                'iso_639_3': input_data['iso_639_3'],
                'wals_language_name': input_data.get('wals_language_name', ''),
                'genre': input_data.get('genre', ''),
                'confidence': input_data.get('confidence', 'low'),
                'wals_features': output_data.get('wals_features', {}),
                'wals_language_id': output_data.get('wals_language_id', '')
            }
        metadata = {}
        print(f"Converted to {len(mappings)} mappings")
    else:
        raise ValueError("Unknown data format")

print(f"\nData loaded successfully!")
print(f"First treebank: {list(mappings.keys())[0]}")


## Convert to Experiment Pipeline Format

The original `data.py` script converts the WALS-UD mapping into the experiment pipeline format. This format is used by downstream evaluation and modeling scripts.

Each UD treebank becomes one **example** with:
- `input`: JSON string describing the treebank
- `output`: JSON string with WALS feature values
- `metadata_*`: Fields for filtering and analysis


In [ ]:
# Convert to experiment format (from data.py, lines 36-78)
# Each UD treebank = one example
examples = []

for tb_name, mapping in mappings.items():
    # Create input text describing the mapping
    input_text = {
        'ud_treebank': mapping['ud_treebank'],
        'ud_language_code': mapping['ud_language_code'],
        'iso_639_3': mapping['iso_639_3'],
        'wals_language_name': mapping.get('wals_language_name', ''),
        'genre': mapping.get('genre', ''),
        'confidence': mapping.get('confidence', 'low')
    }
    
    # Create output with WALS features
    output = {
        'wals_language_id': mapping.get('wals_language_id', ''),
        'wals_features': mapping.get('wals_features', {})
    }
    
    example = {
        'input': json.dumps(input_text, sort_keys=True),
        'output': json.dumps(output, sort_keys=True),
        'metadata_treebank': tb_name,
        'metadata_language_code': mapping['ud_language_code'],
        'metadata_iso_639_3': mapping['iso_639_3'],
        'metadata_confidence': mapping.get('confidence', 'low'),
        'metadata_genre': mapping.get('genre', ''),
        'metadata_num_features': len(mapping.get('wals_features', {}))
    }
    
    examples.append(example)

# Build experiment format
exp_data = {
    'datasets': [
        {
            'dataset': 'wals_ud_mapping',
            'examples': examples
        }
    ]
}

print(f"Created {len(examples)} examples in experiment format")
print(f"\nFirst example (input):")
print(json.dumps(json.loads(examples[0]['input']), indent=2))
print(f"\nFirst example (output):")
print(json.dumps(json.loads(examples[0]['output']), indent=2))


## Results and Visualization

Now let's examine the converted dataset and visualize key properties:
- Confidence level distribution
- WALS feature value distributions
- Language family coverage


In [ ]:
# Summary statistics
print("=" * 60)
print("DATASET SUMMARY")
print("=" * 60)

# Confidence distribution
confidence_counts = Counter(e['metadata_confidence'] for e in examples)
print(f"\nConfidence Levels:")
for conf, count in confidence_counts.most_common():
    print(f"  {conf}: {count} ({100*count/len(examples):.1f}%)")

# Number of features per treebank
feature_counts = [e['metadata_num_features'] for e in examples]
print(f"\nWALS Features per Treebank:")
print(f"  Min: {min(feature_counts)}, Max: {max(feature_counts)}, Avg: {sum(feature_counts)/len(feature_counts):.1f}")

# Genre distribution
genres = [e['metadata_genre'] for e in examples]
print(f"\nGenres:")
for genre in sorted(set(genres)):
    count = genres.count(genre)
    print(f"  {genre}: {count}")

print("\n" + "=" * 60)


In [ ]:
# Visualize the data
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Plot 1: Confidence distribution
conf_labels = list(confidence_counts.keys())
conf_values = list(confidence_counts.values())
axes[0].bar(conf_labels, conf_values, color=['green', 'orange', 'red'])
axes[0].set_title('Mapping Confidence Distribution')
axes[0].set_ylabel('Number of Treebanks')
for i, v in enumerate(conf_values):
    axes[0].text(i, v + 0.1, str(v), ha='center')

# Plot 2: Number of WALS features per treebank
axes[1].hist(feature_counts, bins=5, edgecolor='black', alpha=0.7)
axes[1].set_title('WALS Features per Treebank')
axes[1].set_xlabel('Number of Features')
axes[1].set_ylabel('Count')
axes[1].axvline(sum(feature_counts)/len(feature_counts), color='red', linestyle='--', label='Mean')
axes[1].legend()

plt.tight_layout()
plt.show()

# Print sample WALS feature values
print("\nSample WALS Feature Values:")
for i, ex in enumerate(examples[:3]):
    input_data = json.loads(ex['input'])
    output_data = json.loads(ex['output'])
    print(f"\n{i+1}. {input_data['ud_treebank']} ({input_data['wals_language_name']})")
    for feat_code, feat_info in output_data['wals_features'].items():
        print(f"   {feat_code} ({feat_info['description'][:30]}...): {feat_info['value_description']}")
